In [26]:
import pandas as pd

columns = [
    'age','workclass','fnlwgt','education',
    'education_num','marital_status',
    'occupation','relationship','race',
    'sex','capital_gain','capital_loss',
    'hours_per_week','native_country',
    'income'
]

df = pd.read_csv(
    'adult.data',
    header=None,
    names=columns,
    skipinitialspace=True
)

print(df.head())
print(df.shape)


   age         workclass  fnlwgt  education  education_num  \
0   39         State-gov   77516  Bachelors             13   
1   50  Self-emp-not-inc   83311  Bachelors             13   
2   38           Private  215646    HS-grad              9   
3   53           Private  234721       11th              7   
4   28           Private  338409  Bachelors             13   

       marital_status         occupation   relationship   race     sex  \
0       Never-married       Adm-clerical  Not-in-family  White    Male   
1  Married-civ-spouse    Exec-managerial        Husband  White    Male   
2            Divorced  Handlers-cleaners  Not-in-family  White    Male   
3  Married-civ-spouse  Handlers-cleaners        Husband  Black    Male   
4  Married-civ-spouse     Prof-specialty           Wife  Black  Female   

   capital_gain  capital_loss  hours_per_week native_country income  
0          2174             0              40  United-States  <=50K  
1             0             0             

In [27]:
print(df.info())
print(df['income'].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education_num   32561 non-null  int64 
 5   marital_status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital_gain    32561 non-null  int64 
 11  capital_loss    32561 non-null  int64 
 12  hours_per_week  32561 non-null  int64 
 13  native_country  32561 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB
None
income
<=50K    24720
>50K      7841
Name: count, dtype: int64


In [28]:
import numpy as np

df.replace('?', np.nan, inplace=True)

print(df.isnull().sum())

df.dropna(inplace=True)

print(df.shape)

age                  0
workclass         1836
fnlwgt               0
education            0
education_num        0
marital_status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital_gain         0
capital_loss         0
hours_per_week       0
native_country     583
income               0
dtype: int64
(30162, 15)


In [29]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = le.fit_transform(df[col])

print(df.head())

   age  workclass  fnlwgt  education  education_num  marital_status  \
0   39          5   77516          9             13               4   
1   50          4   83311          9             13               2   
2   38          2  215646         11              9               0   
3   53          2  234721          1              7               2   
4   28          2  338409          9             13               2   

   occupation  relationship  race  sex  capital_gain  capital_loss  \
0           0             1     4    1          2174             0   
1           3             0     4    1             0             0   
2           5             1     4    1             0             0   
3           5             0     2    1             0             0   
4           9             5     2    0             0             0   

   hours_per_week  native_country  income  
0              40              38       0  
1              13              38       0  
2              40   

In [30]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop('income', axis=1)
y = df['income']

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "KNN": KNeighborsClassifier(),
    "SVM": SVC(probability=True)
}

results = []

for name, model in models.items():

    print("Training:", name)

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    y_prob = model.predict_proba(X_test)[:,1]

    results.append([
        name,
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred),
        roc_auc_score(y_test, y_prob)
    ])

import pandas as pd

result_df = pd.DataFrame(
    results,
    columns=[
        "Algorithm",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "ROC-AUC"
    ]
)

result_df

Training: Logistic Regression
Training: Decision Tree
Training: Random Forest
Training: KNN
Training: SVM


,Algorithm,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.823139,0.744456,0.460784,0.569237,0.859774
1,Decision Tree,0.810542,0.626223,0.627451,0.626836,0.750101
2,Random Forest,0.851649,0.738901,0.641830,0.686953,0.908722
3,KNN,0.825791,0.676753,0.599346,0.635702,0.858456
4,SVM,0.847174,0.771914,0.564052,0.651813,0.896074


### Conclusion

In this project, the Adult Census Income dataset was analyzed to predict whether a person's annual income is greater than $50,000. The dataset was first explored to understand its structure and identify missing values. After cleaning the data and encoding categorical features, several machine learning models were trained and tested.

The performance of Logistic Regression, Decision Tree, Random Forest, KNN, and SVM was compared using metrics such as Accuracy, Precision, Recall, F1-Score, and ROC-AUC. Among all the models, Random Forest produced the best overall results and achieved the highest prediction performance on the dataset.

This project helped in understanding the complete machine learning workflow, including data preprocessing, feature engineering, model training, and performance evaluation. It also demonstrated how different algorithms perform on the same dataset and how model comparison can be used to select the most suitable approach for a prediction task.
